In [1]:
import pandas as pd

df = pd.read_csv('/kaggle/input/retail-sales-dataset/retail_sales_dataset.csv')

def rename_col(s):
    s = s.replace(' ', '_')
    s = s.replace('-', '_')
    a = s.split('_')
    for i in range(len(a)):
        if not all([c == c.upper() for c in a[i]]):
            a[i] = a[i].capitalize()
    return '_'.join(a)

df.rename(columns={c: rename_col(c) for c in df.columns}, inplace=True)
df.head()

,Transaction_ID,Date,Customer_ID,Gender,Age,Product_Category,Quantity,Price_Per_Unit,Total_Amount
0,1,2023-11-24,CUST001,Male,34,Beauty,3,50,150
1,2,2023-02-27,CUST002,Female,26,Clothing,2,500,1000
2,3,2023-01-13,CUST003,Male,50,Electronics,1,30,30
3,4,2023-05-21,CUST004,Male,37,Clothing,1,500,500
4,5,2023-05-06,CUST005,Male,30,Beauty,2,50,100


In [2]:
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
df['Month'] = df['Date'].dt.to_period('M').astype(str)

In [3]:
age_bins = [0, 17, 24, 34, 44, 54, 64, 120]
age_labels = ['0-17','18-24','25-34','35-44','45-54','55-64','65+']
df['Age_Group'] = pd.cut(df['Age'], bins=age_bins, labels=age_labels, right=True, include_lowest=True)

In [4]:
df.groupby('Product_Category', dropna=False)['Total_Amount'].sum().sort_values(ascending=False).rename('Total_Sales').reset_index()

,Product_Category,Total_Sales
0,Electronics,156905
1,Clothing,155580
2,Beauty,143515


In [5]:
df.groupby('Product_Category', dropna=False)['Total_Amount'].mean().round(2).sort_values(ascending=False).rename('Avg_Sales_per_Txn').reset_index()

,Product_Category,Avg_Sales_per_Txn
0,Beauty,467.48
1,Electronics,458.79
2,Clothing,443.25


In [6]:
pd.crosstab(df['Product_Category'], df['Age_Group']).reset_index()

Age_Group,Product_Category,18-24,25-34,35-44,45-54,55-64
0,Beauty,53,68,51,73,62
1,Clothing,45,73,79,74,80
2,Electronics,51,62,77,78,74


In [7]:
q4_counts = pd.crosstab(df['Product_Category'], df['Gender']).reset_index()
q4_pct = (pd.crosstab(df['Product_Category'], df['Gender'], normalize='index')
            .mul(100)
            .round(2)
            .add_suffix('_Pct')
            .reset_index())
print(q4_counts)
print(q4_pct)

Gender Product_Category  Female  Male
,0                Beauty     166   141
,1              Clothing     174   177
,2           Electronics     170   172
,Gender Product_Category  Female_Pct  Male_Pct
,0                Beauty       54.07     45.93
,1              Clothing       49.57     50.43
,2           Electronics       49.71     50.29


In [8]:
df.groupby('Age_Group', dropna=False)['Total_Amount'].sum().sort_values(ascending=False).rename('Total_Sales').reset_index()

/tmp/ipykernel_13/1364732883.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
,  df.groupby('Age_Group', dropna=False)['Total_Amount'].sum().sort_values(ascending=False).rename('Total_Sales').reset_index()


,Age_Group,Total_Sales
0,45-54,97235
1,25-34,97090
2,35-44,96835
3,55-64,90190
4,18-24,74650
5,0-17,0
6,65+,0


In [9]:
df.groupby('Product_Category', dropna=False)['Quantity'].agg(['mean','median','sum','count']).round(2).sort_values('mean', ascending=False).reset_index()

,Product_Category,mean,median,sum,count
0,Clothing,2.55,3.0,894,351
1,Beauty,2.51,3.0,771,307
2,Electronics,2.48,2.0,849,342


In [10]:
df.groupby('Month', dropna=False)['Total_Amount'].sum().rename('Total_Sales').reset_index().sort_values('Month')

,Month,Total_Sales
0,2023-01,35450
1,2023-02,44060
2,2023-03,28990
3,2023-04,33870
4,2023-05,53150
5,2023-06,36715
6,2023-07,35465
7,2023-08,36960
8,2023-09,23620
9,2023-10,46580


In [11]:
df.groupby('Customer_ID', dropna=False)['Total_Amount'].sum().sort_values(ascending=False).head(10).rename('Total_Sales').reset_index()

,Customer_ID,Total_Sales
0,CUST487,2000
1,CUST476,2000
2,CUST773,2000
3,CUST503,2000
4,CUST093,2000
5,CUST089,2000
6,CUST946,2000
7,CUST157,2000
8,CUST155,2000
9,CUST420,2000


In [12]:
q9_corr = df[['Quantity','Total_Amount']].corr().round(3).reset_index()
q9_avg_by_qty = (
    df.groupby('Quantity', dropna=False)['Total_Amount']
      .mean()
      .round(2)
      .rename('Avg_Total_Amount')
      .reset_index()
      .sort_values('Quantity')
)
print(q9_corr)
print(q9_avg_by_qty)

          index  Quantity  Total_Amount
,0      Quantity     1.000         0.374
,1  Total_Amount     0.374         1.000
,   Quantity  Avg_Total_Amount
,0         1            177.09
,1         2            333.54
,2         3            598.69
,3         4            706.69


In [13]:
df.groupby('Product_Category', dropna=False)['Price_Per_Unit'].mean().round(2).rename('Avg_Price_Per_Unit').reset_index().sort_values('Avg_Price_Per_Unit', ascending=False)

,Product_Category,Avg_Price_Per_Unit
0,Beauty,184.06
2,Electronics,181.90
1,Clothing,174.29
